In [1]:
from nltk.tokenize import word_tokenize

# test_text = """
#     The quick brown fox jumps over the lazy dog. This is a very common pangram
#     that has been used for testing typewriters and computer keyboards because it
#     contains every single letter of the alphabet. It has been in use since at
#     least the late 1800s and it remains popular today in the modern era.
#     """


test_text = """You are a helpful assistant. Be concise and answer in clear, complete sentences.

    Climate change is one of the most pressing issues facing humanity today. The scientific consensus is clear:
    global temperatures are rising at an unprecedented rate, primarily due to human activities such as burning
    fossil fuels, deforestation, and industrial processes. The Intergovernmental Panel on Climate Change (IPCC)
    has documented extensive evidence showing that the average global temperature has increased by approximately
    1.1°C since the pre-industrial era. This warming trend has led to numerous observable effects, including
    melting polar ice caps, rising sea levels, more frequent and severe weather events, and shifts in wildlife
    populations and habitats.

    The impacts of climate change are far-reaching and affect every aspect of human society and natural ecosystems.
    Coastal communities face increased flooding risks, agricultural systems are disrupted by changing precipitation
    patterns, and extreme weather events cause billions of dollars in damage annually. Scientists warn that without
    significant action to reduce greenhouse gas emissions, these effects will intensify, potentially leading to
    catastrophic consequences for future generations.

    Addressing climate change requires a multifaceted approach involving governments, businesses, and individuals.
    Transitioning to renewable energy sources, improving energy efficiency, protecting and restoring forests, and
    developing sustainable agricultural practices are all crucial steps. International cooperation, as exemplified
    by agreements like the Paris Climate Accord, plays a vital role in coordinating global efforts to limit.
    
    What is the main conclusion of this article in under 100 words?"""

In [2]:
## Caveman Compressor
# This will be our simple baseline compression method which removes certain unnecessary words from the prompt.

import re

stopwords = {
    # Articles
    'a', 'an', 'the',
    # Auxiliary verbs
    'is', 'are', 'was', 'were', 'am', 'be', 'been', 'being',
    'has', 'have', 'had', 'having',
    'do', 'does', 'did', 'doing',
    'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could',
    # Common but often non-essential words
    'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into',
    'through', 'during', 'before', 'after', 'above', 'below', 'from',
    'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again',
    'further', 'then', 'once',
    # Pronouns (keeping some key ones, removing others)
    'this', 'that', 'these', 'those',
    # Other
    'there', 'here', 'where',
    'very', 'just', 'really', 'quite', 'too', 'so',
}

def caveman_compressor(text: str, stopwords: set) -> str:
    """
    Compress text by removing stopwords.
    Preserves sentence structure but creates "caveman" style English.
    """
    # Split into sentences, capturing punctuation
    sentences = re.split(r'([.!?]+)', text)
    
    compressed_sentences = []
    for i in range(0, len(sentences), 2):
        sentence = sentences[i].strip()
        punctuation = sentences[i + 1] if i + 1 < len(sentences) else ''

        # Filter out stopwords
        filtered_words = [
            word for word in sentence.split()
            if re.sub(r'[^\w\']', '', word.lower()) not in stopwords
        ]
        
        if filtered_words:
            compressed_sentences.append(' '.join(filtered_words) + punctuation)
    
    result = ' '.join(compressed_sentences).strip()
    
    # Clean up spacing around punctuation
    result = re.sub(r'\s+([.!?,;:])', r'\1', result)
    result = re.sub(r'\s+', ' ', result)
    
    return result

In [3]:
compressed_text = caveman_compressor(test_text, stopwords)
print(compressed_text)
print()
print(f"Original length (tokens): {len(word_tokenize(test_text.lower()))}")
print(f"Compressed length (tokens): {len(word_tokenize(compressed_text.lower()))}") 

You helpful assistant. concise and answer clear, complete sentences. Climate change one most pressing issues facing humanity today. scientific consensus clear: global temperatures rising unprecedented rate, primarily due to human activities such as burning fossil fuels, deforestation, and industrial processes. Intergovernmental Panel Climate Change (IPCC) documented extensive evidence showing average global temperature increased approximately 1. 1°C since pre-industrial era. warming trend led to numerous observable effects, including melting polar ice caps, rising sea levels, more frequent and severe weather events, and shifts wildlife populations and habitats. impacts climate change far-reaching and affect every aspect human society and natural ecosystems. Coastal communities face increased flooding risks, agricultural systems disrupted changing precipitation patterns, and extreme weather events cause billions dollars damage annually. Scientists warn without significant action to redu

In [4]:
## LLM-Lingua2 Compressor
# This will be our advanced compression method using an LLM to intelligently shorten the prompt while preserving

from llmlingua import PromptCompressor
import torch

# Detect and use the best available backend
if torch.backends.mps.is_available():
    backend = "mps"
    print("Initializing LLMLingua-2 compressor (using MPS - Apple Silicon GPU)...")
elif torch.cuda.is_available():
    backend = "cuda"
    print("Initializing LLMLingua-2 compressor (using CUDA)...")
else:
    backend = "cpu"
    print("Initializing LLMLingua-2 compressor (using CPU)...")

LLMlingua_compressor = PromptCompressor(
    model_name="microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank",
    use_llmlingua2=True,
    device_map=backend,
        )

def LLMlingua_compress(text: str,  rate: float = 0.5, compressor = LLMlingua_compressor) -> str:
    """
    Compress text using LLMLingua-2. 

    Returns:
        Compressed text
    """
    # LLMLingua-2 expects text as a list of strings or single string
    # Only pass target_token if it's set, otherwise use rate

    compress_kwargs = {
        # 'force_tokens': ['\n', '?', '!', '.'],  # Preserve important punctuation
        'use_sentence_level_filter': True,
        'use_context_level_filter': True,
        'use_token_level_filter': True,
        'rate': rate
    }

    result = compressor.compress_prompt(text, **compress_kwargs)

    return result['compressed_prompt']


/Users/adamfletcher/Documents/GitHub/LLM-prompt_compression/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing LLMLingua-2 compressor (using MPS - Apple Silicon GPU)...


`torch_dtype` is deprecated! Use `dtype` instead!


In [5]:
compressed_text = LLMlingua_compress(text= test_text
                   , rate=0.5)

print(compressed_text)
print()
print(f"Original length (tokens): {len(word_tokenize(test_text.lower()))}")
print(f"Compressed length (tokens): {len(word_tokenize(compressed_text.lower()))}") 

helpful assistant concise answer complete sentences Climate change pressing. scientific consensus global temperatures rising unprecedented due to human activities fossil fuels deforestation industrial processes. Intergovernmental Panel on Climate Change documented average global temperature increased 1. 1°C since pre - industrial era warming trend led effects ice caps rising sea levels weather events shifts wildlife populations habitats impacts affect society ecosystems Coastal communities face flooding risks agricultural systems disrupted extreme weather events cause billions damage annually without greenhouse gas emissions effects intensify catastrophic consequences multifaceted approach governments businesses Transitioning renewable energy sources energy efficiency protecting restoring forests sustainable agricultural practices crucial International cooperation Paris Climate Accord global efforts limit main conclusion under 100 words

Original length (tokens): 274
Compressed length 

In [6]:
##Sending Prompts to LLM

from dotenv import load_dotenv
from openai import OpenAI
import os
import tiktoken

load_dotenv()
model = "gpt-4"

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)
encoding = tiktoken.encoding_for_model(model)


prompt = """
Summarize the following text: The quick brown fox jumps over the lazy dog. This is a common pangram used to test typewriters and computer keyboards because it contains every letter of the alphabet. It has been used since at least the late 1800s and remains popular today."
"""

prompt = test_text

def send_prompt(client, prompt: str, **kwargs):
    """Send prompt to OpenAI and return response with token counts."""
    response = client.chat.completions.create(
        model= model,
        messages=[{"role": "user", "content": prompt}],
        **kwargs
    )
    return response.choices[0].message.content


def demo_end_to_end(client, prompt: str):
    """Demonstrate end-to-end compression + LLM."""
    print("\n" + "="*80)
    print("END-TO-END DEMO: COMPRESSION + LLM")
    print("="*80)


    print(f"\nOriginal prompt ({len(word_tokenize(prompt.lower()))} Tokens):")
    print(prompt)

    # Compress with caveman
    print("\n" + "-"*80)
    print("CAVEMAN COMPRESSION")
    print("-"*80)
    caveman_compressed = caveman_compressor(prompt, stopwords)

    print(f"Caveman compressed ({len(word_tokenize(caveman_compressed.lower()))} Tokens):")
    print(caveman_compressed)

    # Compress with LLMLingua-2
    print("\n" + "-"*80)
    print("LLMLINGUA-2 COMPRESSION")
    print("-"*80)

    llmlingua_compressed = LLMlingua_compress(prompt, rate=0.7)

    print(f"LLMLingua-2 compressed ({len(word_tokenize(llmlingua_compressed.lower()))} Tokens):")
    print(llmlingua_compressed)


    # Send to LLM
    print("\n" + "-"*80)
    print("SENDING TO LLM")
    print("-"*80)

    print("\n--- Original prompt ---")
    response1 = send_prompt(client, prompt)
    print(f"Output: {response1}")
    print(f"Tokens used: {len(word_tokenize(response1.lower()))}")

    print("\n--- Caveman compressed prompt ---")
    response2 = send_prompt(client, caveman_compressed)
    print(f"Output: {response2}")
    print(f"Tokens used: {len(word_tokenize(response2.lower()))}")

    print("\n--- LLMLingua-2 compressed prompt ---")
    response3 = send_prompt(client, llmlingua_compressed)

    print(f"Output: {response3}")
    print(f"Tokens used: {len(word_tokenize(response3.lower()))}")


In [7]:
demo_end_to_end(client, prompt)


END-TO-END DEMO: COMPRESSION + LLM

Original prompt (274 Tokens):
You are a helpful assistant. Be concise and answer in clear, complete sentences.

    Climate change is one of the most pressing issues facing humanity today. The scientific consensus is clear:
    global temperatures are rising at an unprecedented rate, primarily due to human activities such as burning
    fossil fuels, deforestation, and industrial processes. The Intergovernmental Panel on Climate Change (IPCC)
    has documented extensive evidence showing that the average global temperature has increased by approximately
    1.1°C since the pre-industrial era. This warming trend has led to numerous observable effects, including
    melting polar ice caps, rising sea levels, more frequent and severe weather events, and shifts in wildlife
    populations and habitats.

    The impacts of climate change are far-reaching and affect every aspect of human society and natural ecosystems.
    Coastal communities face increas

In [8]:
## Oreo Method: Compress the middle bit around some key start and end instructions

prompt = {
    'system_instructions': "You are a helpful assistant. Be concise and answer in clear, complete sentences.",
    'content': """
    Climate change is one of the most pressing issues facing humanity today. The scientific consensus is clear:
    global temperatures are rising at an unprecedented rate, primarily due to human activities such as burning
    fossil fuels, deforestation, and industrial processes. The Intergovernmental Panel on Climate Change (IPCC)
    has documented extensive evidence showing that the average global temperature has increased by approximately
    1.1°C since the pre-industrial era. This warming trend has led to numerous observable effects, including
    melting polar ice caps, rising sea levels, more frequent and severe weather events, and shifts in wildlife
    populations and habitats.

    The impacts of climate change are far-reaching and affect every aspect of human society and natural ecosystems.
    Coastal communities face increased flooding risks, agricultural systems are disrupted by changing precipitation
    patterns, and extreme weather events cause billions of dollars in damage annually. Scientists warn that without
    significant action to reduce greenhouse gas emissions, these effects will intensify, potentially leading to
    catastrophic consequences for future generations.

    Addressing climate change requires a multifaceted approach involving governments, businesses, and individuals.
    Transitioning to renewable energy sources, improving energy efficiency, protecting and restoring forests, and
    developing sustainable agricultural practices are all crucial steps. International cooperation, as exemplified
    by agreements like the Paris Climate Accord, plays a vital role in coordinating global efforts to limit
    temperature increases and adapt to unavoidable changes.
    """,
    'query': "What is the main conclusion of this article in under 100 words?"}



def oreo_demo(client, prompt: str):
    """Demonstrate end-to-end compression + LLM."""
    print("\n" + "="*80)
    print("OREO DEMO: COMPRESSION + LLM")
    print("="*80)

    original_prompt = prompt["system_instructions"] + "\n" + prompt["content"] + "\n" + prompt["query"]
    print(original_prompt)
    print(f"\nOriginal prompt ({len(word_tokenize(original_prompt.lower()))} Tokens):")

    # Compress with caveman
    print("\n" + "-"*80)
    print("CAVEMAN COMPRESSION")
    print("-"*80)
    caveman_compressed = caveman_compressor(prompt["content"], stopwords)
    oreo_caveman_prompt = prompt["system_instructions"] + "\n" + caveman_compressed + "\n" + prompt["query"]

    print(f"Caveman compressed ({len(word_tokenize(caveman_compressed.lower()))} Tokens):")
    print(oreo_caveman_prompt)

    # Compress with LLMLingua-2
    print("\n" + "-"*80)
    print("LLMLINGUA-2 COMPRESSION")
    print("-"*80)

    llmlingua_compressed = LLMlingua_compress(prompt["content"], rate=0.5)
    oreo_llmlingua_prompt = prompt["system_instructions"] + "\n" + llmlingua_compressed + "\n" + prompt["query"]

    print(f"LLMLingua-2 compressed ({len(word_tokenize(llmlingua_compressed.lower()))} Tokens):")
    print(oreo_llmlingua_prompt)


    # Send to LLM
    print("\n" + "-"*80)
    print("SENDING TO LLM")
    print("-"*80)

    print("\n--- Original prompt ---")
    response1 = send_prompt(client, original_prompt)
    print(f"Output: {response1}")
    print(f"Tokens used: {len(word_tokenize(response1.lower()))}")

    print("\n--- Caveman compressed prompt ---")
    response2 = send_prompt(client, oreo_caveman_prompt)
    print(f"Output: {response2}")
    print(f"Tokens used: {len(word_tokenize(response2.lower()))}")

    print("\n--- LLMLingua-2 compressed prompt ---")
    response3 = send_prompt(client, oreo_llmlingua_prompt)
    print(f"Output: {response3}")
    print(f"Tokens used: {len(word_tokenize(response3.lower()))}")

In [9]:
oreo_demo(client, prompt)


OREO DEMO: COMPRESSION + LLM
You are a helpful assistant. Be concise and answer in clear, complete sentences.

    Climate change is one of the most pressing issues facing humanity today. The scientific consensus is clear:
    global temperatures are rising at an unprecedented rate, primarily due to human activities such as burning
    fossil fuels, deforestation, and industrial processes. The Intergovernmental Panel on Climate Change (IPCC)
    has documented extensive evidence showing that the average global temperature has increased by approximately
    1.1°C since the pre-industrial era. This warming trend has led to numerous observable effects, including
    melting polar ice caps, rising sea levels, more frequent and severe weather events, and shifts in wildlife
    populations and habitats.

    The impacts of climate change are far-reaching and affect every aspect of human society and natural ecosystems.
    Coastal communities face increased flooding risks, agricultural syste

In [10]:
## The combination of using A compressor with the Oreo method PLUS an output response length seems to give the best results overall.

# But, it means that it's use it more specific to general purpose. 